# NUS ST4250 — Multivariate Statistical Analysis
## A systematic, intuitive, case-study notebook with Bokeh

This notebook is designed as a **student-facing companion** to the central ideas of NUS ST4250. Rather than treating PCA, MANOVA, discriminant analysis and clustering as isolated algorithms, we build them from one common object:

$$
\mathbf X=(X_1,\ldots,X_p)^T.
$$

The guiding question is:

> **How do we reason statistically when every observation is a vector rather than a single number?**

We use the Wine dataset bundled with scikit-learn. Each wine is represented by 13 continuous chemical measurements and belongs to one of three cultivars. That gives us a compact but realistic setting for studying covariance, multivariate inference, dimensionality reduction, classification and clustering.

### Learning objectives

By the end, you should be able to:

1. represent and inspect multivariate data as an $n\times p$ matrix;
2. interpret mean vectors, covariance matrices and correlation matrices geometrically;
3. understand eigenvalues/eigenvectors as directions and magnitudes of variation;
4. distinguish Euclidean distance from Mahalanobis distance;
5. understand the multivariate normal model and a practical normality diagnostic;
6. carry out and interpret a two-sample Hotelling $T^2$ test;
7. understand what MANOVA tests jointly that separate ANOVAs do not;
8. derive PCA from the covariance/correlation matrix and compare it with scikit-learn;
9. understand why scaling can materially change PCA;
10. use LDA as a model-based classification method;
11. contrast supervised classification with unsupervised clustering;
12. evaluate clustering without accidentally using labels during training;
13. connect the entire workflow into one ST4250 mental model.

> **Important:** The notebook intentionally alternates between mathematics, code, visualisation and interpretation. Do not rush through the code cells; the interpretation cells are part of the lesson.

## 0. Concept map

A useful map of the module is

$$
\text{multivariate data}
\rightarrow
\text{mean/covariance structure}
\rightarrow
\text{multivariate probability}
\rightarrow
\text{joint inference}
\rightarrow
\begin{cases}
\text{dimension reduction}\\
\text{classification}\\
\text{clustering}
\end{cases}
$$

The same covariance matrix $\Sigma$ reappears throughout:

- in the multivariate normal density;
- in Mahalanobis distance;
- in Hotelling's $T^2$;
- in PCA through eigen-decomposition;
- in LDA through class-conditional Gaussian models.

That repetition is not accidental. **Covariance is the geometry of multivariate statistics.**

## 1. Environment and imports

The notebook uses only common scientific Python libraries. Bokeh is used for all visualisations. The plotting helpers below avoid fragile Bokeh patterns such as `legend.location = "best"`, which is not a valid modern Bokeh legend location.

In [1]:
# Uncomment only if your environment is missing packages.
# %pip install -q numpy pandas scipy scikit-learn statsmodels bokeh

from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, Sequence
import math
import re

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    silhouette_score,
    adjusted_rand_score,
)
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.pipeline import Pipeline

from statsmodels.multivariate.manova import MANOVA

from bokeh.io import output_notebook, show
from bokeh.layouts import gridplot, column
from bokeh.models import (
    ColumnDataSource,
    DataTable,
    TableColumn,
    HoverTool,
    LinearColorMapper,
    ColorBar,
    BasicTicker,
    Span,
)
from bokeh.palettes import Category10, Viridis256
from bokeh.plotting import figure
from bokeh.transform import transform

output_notebook(hide_banner=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 30)
pd.set_option('display.precision', 4)

## 2. Reusable helpers

A statistical notebook becomes easier to reason about when plotting and numerical utilities are separated from the analysis logic.

We use two small helper classes:

- `PlotFactory`: Bokeh visualisations and tables;
- `MultivariateMath`: reusable statistical calculations.

The code is deliberately explicit rather than compressed into one-liners so that students can inspect each transformation.

In [2]:
@dataclass(frozen=True)
class PlotFactory:
    width: int = 720
    height: int = 420

    @staticmethod
    def _safe_df(df: pd.DataFrame) -> pd.DataFrame:
        """Return a Bokeh-friendly dataframe with simple string column names."""
        x = df.copy().reset_index(drop=True)
        x.columns = [str(c) for c in x.columns]
        for c in x.columns:
            if pd.api.types.is_float_dtype(x[c]):
                x[c] = x[c].round(4)
        return x

    def table(self, df: pd.DataFrame, title: str, width: int | None = None, height: int = 250):
        x = self._safe_df(df)
        source = ColumnDataSource(x)
        columns = [TableColumn(field=c, title=c.replace('_', ' ').title()) for c in x.columns]
        table = DataTable(source=source, columns=columns, width=width or self.width, height=height, index_position=None)
        display(Markdown(f"**{title}**"))
        show(table)

    def heatmap(self, matrix: pd.DataFrame, title: str, low: float | None = None, high: float | None = None):
        long = (
            matrix.rename_axis(index='row', columns='column')
                  .stack()
                  .rename('value')
                  .reset_index()
        )
        low = float(long['value'].min()) if low is None else low
        high = float(long['value'].max()) if high is None else high
        mapper = LinearColorMapper(palette=Viridis256, low=low, high=high)
        p = figure(
            x_range=[str(c) for c in matrix.columns],
            y_range=[str(i) for i in matrix.index[::-1]],
            width=max(self.width, 760),
            height=max(self.height, 520),
            title=title,
            toolbar_location='above',
            tools='pan,wheel_zoom,box_zoom,reset,save',
        )
        src = ColumnDataSource(long.assign(row=long['row'].astype(str), column=long['column'].astype(str)))
        p.rect(
            x='column', y='row', width=1, height=1,
            source=src,
            line_color=None,
            fill_color=transform('value', mapper),
        )
        p.add_tools(HoverTool(tooltips=[('row', '@row'), ('column', '@column'), ('value', '@value{0.0000}')]))
        p.add_layout(ColorBar(color_mapper=mapper, ticker=BasicTicker(), label_standoff=8), 'right')
        p.xaxis.major_label_orientation = math.pi / 3
        return p

    def grouped_scatter(self, df: pd.DataFrame, x: str, y: str, group: str, title: str, hover_cols: Sequence[str] = ()):
        p = figure(
            width=self.width, height=self.height, title=title,
            tools='pan,wheel_zoom,box_zoom,reset,save', active_scroll='wheel_zoom'
        )
        groups = list(pd.unique(df[group]))
        palette = Category10[10]
        for i, g in enumerate(groups):
            sub = df[df[group] == g].copy()
            src = ColumnDataSource(sub)
            p.scatter(
                x=x, y=y, source=src, size=8, alpha=0.72,
                color=palette[i % len(palette)], legend_label=str(g),
            )
        tooltips = [(x, f'@{x}{{0.000}}'), (y, f'@{y}{{0.000}}'), (group, f'@{group}')]
        tooltips += [(c, f'@{c}') for c in hover_cols]
        p.add_tools(HoverTool(tooltips=tooltips))
        p.legend.location = 'top_left'
        p.legend.click_policy = 'hide'
        p.xaxis.axis_label = x
        p.yaxis.axis_label = y
        return p

    def line(self, x, y, title: str, x_label: str, y_label: str, points: bool = True):
        p = figure(width=self.width, height=self.height, title=title, tools='pan,wheel_zoom,box_zoom,reset,save')
        p.line(x, y, line_width=2)
        if points:
            p.scatter(x, y, size=7)
        p.xaxis.axis_label = x_label
        p.yaxis.axis_label = y_label
        return p


class MultivariateMath:
    @staticmethod
    def sample_mean_cov(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        X = np.asarray(X, dtype=float)
        return X.mean(axis=0), np.cov(X, rowvar=False, ddof=1)

    @staticmethod
    def mahalanobis_squared(X: np.ndarray, mean: np.ndarray, cov: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        delta = X - mean
        precision = np.linalg.pinv(cov)
        return np.einsum('ij,jk,ik->i', delta, precision, delta)

    @staticmethod
    def explained_variance_from_cov(cov: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        eigvals, eigvecs = np.linalg.eigh(cov)
        order = np.argsort(eigvals)[::-1]
        eigvals = eigvals[order]
        eigvecs = eigvecs[:, order]
        ratio = eigvals / eigvals.sum()
        return ratio, eigvecs


PLOT = PlotFactory()
MATH = MultivariateMath()

## 3. Load the case-study dataset

The Wine dataset contains $n=178$ wines and $p=13$ continuous chemical measurements. The target contains three cultivars.

This is useful pedagogically because the **measurements are multivariate**, while the class labels let us later compare:

- unsupervised structure discovered from $X$ alone;
- supervised structure learned using known classes.

The labels will **not** be used to fit PCA or clustering unless explicitly stated.

In [3]:
raw = load_wine(as_frame=True)

def safe_name(name: str) -> str:
    name = re.sub(r'[^0-9a-zA-Z_]+', '_', name.strip().lower())
    return re.sub(r'_+', '_', name).strip('_')

X = raw.data.copy()
X.columns = [safe_name(c) for c in X.columns]
y = raw.target.copy()

class_map = {i: f'cultivar_{i + 1}' for i in sorted(y.unique())}
df = X.copy()
df['class_id'] = y.astype(int)
df['class_label'] = y.map(class_map)

print(f'n observations = {len(df)}')
print(f'p measurements = {X.shape[1]}')
display(df.head())

n observations = 178
p measurements = 13


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280_od315_of_diluted_wines,proline,class_id,class_label
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0,cultivar_1
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0,cultivar_1
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0,cultivar_1
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0,cultivar_1
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0,cultivar_1


### What is the multivariate object here?

For observation $i$,

$$
\mathbf x_i = (x_{i1},x_{i2},\ldots,x_{i13})^T.
$$

Stacking all observations gives an $178\times13$ data matrix:

$$
X=
\begin{bmatrix}
\mathbf x_1^T\\
\mathbf x_2^T\\
\vdots\\
\mathbf x_{178}^T
\end{bmatrix}.
$$

A central theme of ST4250 is that we should often analyse this matrix **jointly**, rather than running 13 disconnected univariate analyses.

In [60]:
summary = X.describe().T.reset_index().rename(columns={'index': 'feature'})
PLOT.table(summary[['feature', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']],
           'Univariate summaries — useful, but not yet multivariate', height=330)

**Univariate summaries — useful, but not yet multivariate**

## 4. Why scaling matters before multivariate analysis

Look at the standard deviations above. Some measurements naturally have much larger numerical scales than others.

If we define

$$
Z_j = \frac{X_j-\bar X_j}{s_j},
$$

then each standardized feature has approximately mean 0 and variance 1.

This is not a cosmetic transformation. Many multivariate methods are sensitive to scale because they depend on distances or variances.

We therefore keep both versions:

- `X_raw`: original units;
- `X_std`: standardized variables.

In [61]:
scaler = StandardScaler()
X_raw = X.to_numpy(dtype=float)
X_std = scaler.fit_transform(X_raw)

scale_check = pd.DataFrame({
    'feature': X.columns,
    'raw_mean': X_raw.mean(axis=0),
    'raw_sd': X_raw.std(axis=0, ddof=1),
    'std_mean': X_std.mean(axis=0),
    'std_sd': X_std.std(axis=0, ddof=1),
})
PLOT.table(scale_check, 'Effect of standardization', height=330)

**Effect of standardization**

### Interpretation

After standardization, all columns contribute on a comparable numerical scale. This becomes especially important for:

- PCA;
- Euclidean-distance clustering;
- visual comparisons of coefficients/loadings.

However, standardization can also remove meaningful scale information. Therefore the correct question is not *"Should I always scale?"* but rather:

> **Does the numerical unit itself carry information I want the method to respect?**

# Part I — Mean vectors and covariance geometry

## 5. The sample mean vector

For $p$ variables, the sample mean is not a scalar but a vector:

$$
\bar{\mathbf x}=\frac{1}{n}\sum_{i=1}^{n}\mathbf x_i.
$$

It is the multivariate centre of the sample.

In [34]:
mean_vec, cov_mat = MATH.sample_mean_cov(X_raw)
mean_df = pd.DataFrame({'feature': X.columns, 'sample_mean': mean_vec})
PLOT.table(mean_df, 'Sample mean vector', height=330)

**Sample mean vector**

## 6. The covariance matrix

For centered observations, the sample covariance matrix is

$$
S = \frac{1}{n-1}\sum_{i=1}^{n}(\mathbf x_i-\bar{\mathbf x})(\mathbf x_i-\bar{\mathbf x})^T.
$$

Its diagonal entries are variances. Its off-diagonal entries are covariances.

A useful mental model is:

> **The mean vector tells us where the cloud is; the covariance matrix tells us its shape and orientation.**

In [35]:
cov_df = pd.DataFrame(cov_mat, index=X.columns, columns=X.columns)
show(PLOT.heatmap(cov_df, 'Sample covariance matrix'))

### Why the covariance heatmap can be misleading

Covariance depends on measurement units. A feature measured in large numerical units can produce large covariances simply because of scale.

For cross-feature comparison we often inspect the correlation matrix

$$
R_{jk}=\frac{S_{jk}}{\sqrt{S_{jj}S_{kk}}}.
$$

Every correlation lies between $-1$ and $1$.

In [62]:
corr_df = X.corr()
show(PLOT.heatmap(corr_df, 'Correlation matrix', low=-1, high=1))

### Questions to ask while reading the correlation matrix

1. Which variables are strongly positively related?
2. Which variables are strongly negatively related?
3. Are there groups of features carrying similar information?
4. Might a lower-dimensional representation capture much of the structure?

That last question is exactly what motivates PCA later.

## 7. Covariance as geometry: a two-variable example

To see covariance geometrically, take two standardized variables. If the cloud is elongated along a diagonal, the variables co-vary. The principal axes of the cloud are the eigenvectors of its covariance matrix.

In [63]:
feat_x, feat_y = 'flavanoids', 'od280_od315_of_diluted_wines'
pair = pd.DataFrame({
    feat_x: X_std[:, X.columns.get_loc(feat_x)],
    feat_y: X_std[:, X.columns.get_loc(feat_y)],
    'class_label': df['class_label'],
})

S2 = pair[[feat_x, feat_y]].cov().to_numpy()
mu2 = pair[[feat_x, feat_y]].mean().to_numpy()
eigvals2, eigvecs2 = np.linalg.eigh(S2)
order = np.argsort(eigvals2)[::-1]
eigvals2, eigvecs2 = eigvals2[order], eigvecs2[:, order]

p = PLOT.grouped_scatter(pair, feat_x, feat_y, 'class_label', 'Covariance geometry in two dimensions')

# Draw principal axes through the sample mean.
for k in range(2):
    v = eigvecs2[:, k]
    length = 2.5 * np.sqrt(eigvals2[k])
    x0, y0 = mu2 - length * v
    x1, y1 = mu2 + length * v
    p.line([x0, x1], [y0, y1], line_width=4 - k, alpha=0.8)

show(p)

axis_table = pd.DataFrame({
    'axis': ['principal_axis_1', 'principal_axis_2'],
    'eigenvalue': eigvals2,
    f'loading_{feat_x}': eigvecs2[0],
    f'loading_{feat_y}': eigvecs2[1],
})
PLOT.table(axis_table, 'Eigenvalues and eigenvectors of the 2D covariance matrix', height=180)

**Eigenvalues and eigenvectors of the 2D covariance matrix**

### Interpretation

The longest principal axis points in the direction of greatest sample variance. Its eigenvalue measures how much variance lies along that direction.

This gives the key PCA relationship:

$$
S\mathbf v_k=\lambda_k\mathbf v_k.
$$

- $\mathbf v_k$: direction;
- $\lambda_k$: variance in that direction.

PCA will simply extend this geometry from 2 dimensions to 13 dimensions.

# Part II — Random vectors and the multivariate normal model

## 8. From a random variable to a random vector

Univariate probability studies $X$. Multivariate probability studies

$$
\mathbf X=(X_1,\ldots,X_p)^T.
$$

Its expectation and covariance are

$$
E(\mathbf X)=\boldsymbol\mu,
$$

$$
\operatorname{Cov}(\mathbf X)=\Sigma.
$$

For a multivariate normal vector,

$$
\mathbf X\sim N_p(\boldsymbol\mu,\Sigma).
$$

The covariance matrix now directly controls the shape of probability contours.

In [38]:
def simulate_bivariate(rho: float, n: int = 500) -> pd.DataFrame:
    cov = np.array([[1.0, rho], [rho, 1.0]])
    z = np.random.multivariate_normal([0, 0], cov, size=n)
    return pd.DataFrame({'x1': z[:, 0], 'x2': z[:, 1], 'rho': f'rho={rho:+.1f}'})

panels = []
for rho in (-0.8, 0.0, 0.8):
    sim = simulate_bivariate(rho)
    p = figure(width=330, height=320, title=f'Bivariate normal: rho={rho:+.1f}', tools='pan,wheel_zoom,reset')
    p.scatter(sim['x1'], sim['x2'], size=5, alpha=0.45)
    p.xaxis.axis_label = 'X1'
    p.yaxis.axis_label = 'X2'
    panels.append(p)
show(gridplot([panels]))

### What changed?

The marginal variances remained fixed at 1. Only the covariance changed.

- $\rho\approx0$: nearly circular cloud;
- $\rho>0$: upward-sloping ellipse;
- $\rho<0$: downward-sloping ellipse.

So covariance is not just an entry in a table. It is **visible geometry**.

## 9. Mahalanobis distance

Euclidean distance treats all directions as equally variable:

$$
d_E^2(\mathbf x,\boldsymbol\mu)
=(\mathbf x-\boldsymbol\mu)^T(\mathbf x-\boldsymbol\mu).
$$

Mahalanobis distance adjusts for covariance:

$$
d_M^2(\mathbf x,\boldsymbol\mu)
=(\mathbf x-\boldsymbol\mu)^T\Sigma^{-1}(\mathbf x-\boldsymbol\mu).
$$

An observation far away along a naturally high-variance direction may be less surprising than an observation only moderately far away in a very stable direction.

In [39]:
mean_std, cov_std = MATH.sample_mean_cov(X_std)
md2 = MATH.mahalanobis_squared(X_std, mean_std, cov_std)
euclid2 = np.sum((X_std - mean_std) ** 2, axis=1)

# Under an ideal p-dimensional normal model, squared Mahalanobis distance is approximately chi-square_p.
p_dim = X_std.shape[1]
threshold_975 = stats.chi2.ppf(0.975, df=p_dim)

outlier_df = pd.DataFrame({
    'row': np.arange(len(df)),
    'class_label': df['class_label'],
    'euclidean_sq': euclid2,
    'mahalanobis_sq': md2,
    'flag_97_5pct': md2 > threshold_975,
}).sort_values('mahalanobis_sq', ascending=False)

PLOT.table(outlier_df.head(15), 'Largest covariance-adjusted distances', height=360)

p = figure(width=760, height=430, title='Euclidean distance vs Mahalanobis distance',
           tools='pan,wheel_zoom,box_zoom,reset,save')
p.scatter('euclidean_sq', 'mahalanobis_sq', source=ColumnDataSource(outlier_df), size=7, alpha=0.65)
p.add_layout(Span(location=threshold_975, dimension='width', line_dash='dashed', line_width=2))
p.xaxis.axis_label = 'Squared Euclidean distance in standardized space'
p.yaxis.axis_label = 'Squared Mahalanobis distance'
p.add_tools(HoverTool(tooltips=[('row', '@row'), ('class', '@class_label'), ('M^2', '@mahalanobis_sq{0.00}')]))
show(p)

print(f'Chi-square 97.5% reference threshold with df={p_dim}: {threshold_975:.3f}')
print(f'Flagged observations: {(md2 > threshold_975).sum()} of {len(md2)}')

**Largest covariance-adjusted distances**

Chi-square 97.5% reference threshold with df=13: 24.736
Flagged observations: 12 of 178


### Important caution

The chi-square threshold above is a **model-based diagnostic**, not an automatic outlier truth machine.

Its validity depends on the multivariate normal approximation and on estimating $\mu$ and $\Sigma$ adequately. In real analysis, flagged observations should be investigated, not blindly deleted.

## 10. A multivariate normality diagnostic using Mahalanobis distances

If

$$
\mathbf X\sim N_p(\boldsymbol\mu,\Sigma),
$$

then approximately

$$
d_M^2\sim\chi^2_p.
$$

Therefore we can compare ordered squared Mahalanobis distances with theoretical $\chi^2_p$ quantiles. A roughly straight relationship supports the model; systematic curvature or extreme tail departures suggest deviations.

In [40]:
ordered_md2 = np.sort(md2)
n = len(ordered_md2)
probs = (np.arange(1, n + 1) - 0.5) / n
chi_quantiles = stats.chi2.ppf(probs, df=p_dim)

qq = pd.DataFrame({'theoretical_chi2': chi_quantiles, 'observed_md2': ordered_md2})
limit = max(qq.max())
p = figure(width=720, height=430, title='Multivariate normality diagnostic: Mahalanobis Q-Q plot',
           tools='pan,wheel_zoom,box_zoom,reset,save')
p.scatter('theoretical_chi2', 'observed_md2', source=ColumnDataSource(qq), size=6, alpha=0.65)
p.line([0, limit], [0, limit], line_dash='dashed', line_width=2)
p.xaxis.axis_label = f'Theoretical chi-square quantiles (df={p_dim})'
p.yaxis.axis_label = 'Ordered squared Mahalanobis distances'
show(p)

### How to read the Q-Q plot

- close to the reference line: broadly compatible with multivariate normal geometry;
- upward tail deviation: heavier-than-normal multivariate tails / unusual observations;
- strong curvature: the Gaussian covariance model may not describe the sample well.

This is more informative than checking 13 marginal histograms independently because **joint normality is a multivariate property**.

# Part III — Multivariate inference

## 11. Why separate t-tests are not the same as a multivariate test

Suppose two cultivars differ across several chemical measurements. Running one t-test per variable creates two issues:

1. repeated testing inflates the chance of false positives;
2. separate tests ignore covariance among outcomes.

Hotelling's $T^2$ asks one joint question:

$$
H_0:\boldsymbol\mu_1=\boldsymbol\mu_2.
$$

For two independent groups with a pooled covariance matrix,

$$
T^2=\frac{n_1n_2}{n_1+n_2}
(\bar{\mathbf x}_1-\bar{\mathbf x}_2)^T
S_p^{-1}
(\bar{\mathbf x}_1-\bar{\mathbf x}_2).
$$

In [41]:
@dataclass(frozen=True)
class HotellingResult:
    t2: float
    f_stat: float
    df1: int
    df2: int
    p_value: float


def hotelling_two_sample(X1: np.ndarray, X2: np.ndarray) -> HotellingResult:
    X1 = np.asarray(X1, dtype=float)
    X2 = np.asarray(X2, dtype=float)
    n1, p = X1.shape
    n2, p2 = X2.shape
    if p != p2:
        raise ValueError('Groups must have the same number of variables.')
    if n1 + n2 <= p + 1:
        raise ValueError('Not enough observations for the classical F conversion.')

    m1, m2 = X1.mean(axis=0), X2.mean(axis=0)
    S1 = np.cov(X1, rowvar=False, ddof=1)
    S2 = np.cov(X2, rowvar=False, ddof=1)
    Sp = ((n1 - 1) * S1 + (n2 - 1) * S2) / (n1 + n2 - 2)

    delta = m1 - m2
    t2 = (n1 * n2 / (n1 + n2)) * (delta @ np.linalg.pinv(Sp) @ delta)

    df1 = p
    df2 = n1 + n2 - p - 1
    f_stat = ((n1 + n2 - p - 1) / (p * (n1 + n2 - 2))) * t2
    p_value = stats.f.sf(f_stat, df1, df2)
    return HotellingResult(float(t2), float(f_stat), df1, df2, float(p_value))


# Use a moderate subset of variables so the geometry and assumptions are easier to inspect.
inference_features = ['alcohol', 'malic_acid', 'flavanoids', 'color_intensity', 'proline']
X_inf = df[inference_features + ['class_id']]
g1 = X_inf.loc[X_inf['class_id'] == 0, inference_features].to_numpy()
g2 = X_inf.loc[X_inf['class_id'] == 1, inference_features].to_numpy()

hotelling = hotelling_two_sample(g1, g2)
hotelling

HotellingResult(t2=560.8946261508428, f_stat=108.67333381672579, df1=5, df2=124, p_value=1.3425827719985094e-43)

In [42]:
hotelling_table = pd.DataFrame([{
    'comparison': 'cultivar_1 vs cultivar_2',
    'variables': len(inference_features),
    'T2': hotelling.t2,
    'F': hotelling.f_stat,
    'df1': hotelling.df1,
    'df2': hotelling.df2,
    'p_value': hotelling.p_value,
}])
PLOT.table(hotelling_table, 'Two-sample Hotelling T-squared result', height=150)

**Two-sample Hotelling T-squared result**

### Interpretation

The p-value answers a **joint** question about the five-dimensional mean vectors. If it is very small, the evidence is inconsistent with equal population mean vectors.

Notice what the test does **not** tell us: it does not directly identify which variables drive the difference. Follow-up univariate or simultaneous confidence-interval analysis is needed for interpretation.

### Classical assumptions

The standard two-sample Hotelling test relies on assumptions analogous to the pooled two-sample t-test:

- independent observations;
- approximate multivariate normality within groups;
- common covariance matrix across the two populations.

When covariance matrices differ substantially, robust or alternative multivariate procedures may be more appropriate.

## 12. MANOVA: several groups and several outcomes

ANOVA tests whether group means differ for one response.

MANOVA tests whether groups differ in a **vector of responses**.

For three cultivars and four selected measurements, conceptually:

$$
H_0:\boldsymbol\mu_1=\boldsymbol\mu_2=\boldsymbol\mu_3.
$$

MANOVA works with the separation of between-group and within-group variation in matrix form. Common test statistics include Wilks' lambda, Pillai's trace, Hotelling-Lawley trace and Roy's largest root.

In [43]:
manova_features = ['alcohol', 'flavanoids', 'color_intensity', 'proline']
formula = ' + '.join(manova_features) + ' ~ C(class_label)'
manova_model = MANOVA.from_formula(formula, data=df)
manova_result = manova_model.mv_test()
print(manova_result)

                    Multivariate linear model
                                                                  
------------------------------------------------------------------
       Intercept         Value   Num DF  Den DF   F Value   Pr > F
------------------------------------------------------------------
          Wilks' lambda   0.0037 4.0000 172.0000 11583.0235 0.0000
         Pillai's trace   0.9963 4.0000 172.0000 11583.0235 0.0000
 Hotelling-Lawley trace 269.3726 4.0000 172.0000 11583.0235 0.0000
    Roy's greatest root 269.3726 4.0000 172.0000 11583.0235 0.0000
------------------------------------------------------------------
                                                                  
------------------------------------------------------------------
         C(class_label)     Value  Num DF  Den DF  F Value  Pr > F
------------------------------------------------------------------
              Wilks' lambda 0.0372 8.0000 344.0000 180.0787 0.0000
             Pil

### What should a student focus on in MANOVA output?

Do not get lost in the four named statistics immediately. First ask:

1. What is the null hypothesis?
2. Is the joint group effect statistically detectable?
3. Are the response variables correlated enough that a joint analysis is scientifically sensible?
4. What follow-up analysis is needed to understand the direction of the differences?

A significant MANOVA is usually the beginning of interpretation, not the end.

In [44]:
# Standardized group means provide an intuitive effect-profile view after MANOVA.
zdf = pd.DataFrame(X_std, columns=X.columns)
zdf['class_label'] = df['class_label'].to_numpy()
profile = zdf.groupby('class_label')[manova_features].mean().T
profile.index.name = 'feature'
profile_long = profile.reset_index().melt(id_vars='feature', var_name='class_label', value_name='standardized_mean')

p = figure(x_range=manova_features, width=780, height=430, title='Post-MANOVA interpretation: standardized group mean profiles',
           tools='pan,wheel_zoom,reset,save')
for i, g in enumerate(profile.columns):
    sub = profile_long[profile_long['class_label'] == g]
    p.line(sub['feature'], sub['standardized_mean'], line_width=2, legend_label=str(g), color=Category10[10][i])
    p.scatter(sub['feature'], sub['standardized_mean'], size=8, color=Category10[10][i])
p.legend.location = 'top_left'
p.xaxis.major_label_orientation = math.pi / 4
p.yaxis.axis_label = 'Group mean in standardized units'
show(p)

# Part IV — Principal Component Analysis

## 13. PCA intuition before formulas

Suppose 13 variables are strongly correlated. Carrying all 13 axes may be redundant.

PCA rotates the coordinate system so that:

- PC1 points in the direction of greatest variance;
- PC2 captures the greatest remaining variance subject to orthogonality to PC1;
- and so on.

For standardized data with covariance/correlation matrix $S$,

$$
S\mathbf v_k=\lambda_k\mathbf v_k.
$$

The component score for observation $i$ is

$$
z_{ik}=\mathbf v_k^T\mathbf x_i.
$$

The explained-variance proportion is

$$
\frac{\lambda_k}{\sum_{j=1}^p\lambda_j}.
$$

## 14. PCA from scratch using eigen-decomposition

Because standardized variables have unit variance, the covariance matrix of `X_std` is approximately the correlation matrix of the raw variables.

In [17]:
S_std = np.cov(X_std, rowvar=False, ddof=1)
ratio_scratch, V_scratch = MATH.explained_variance_from_cov(S_std)

scores_scratch = X_std @ V_scratch

pca_scratch_summary = pd.DataFrame({
    'component': np.arange(1, len(ratio_scratch) + 1),
    'explained_variance_ratio': ratio_scratch,
    'cumulative_variance': np.cumsum(ratio_scratch),
})
PLOT.table(pca_scratch_summary, 'PCA from covariance eigen-decomposition', height=360)

**PCA from covariance eigen-decomposition**

## 15. Verify against scikit-learn

A useful scientific-programming habit is to derive a method once and then verify it against a trusted implementation.

Eigenvectors are only defined up to sign, so a component may appear multiplied by $-1$ without changing the PCA solution.

In [45]:
pca = PCA()
X_pca = pca.fit_transform(X_std)

verification = pd.DataFrame({
    'component': np.arange(1, X.shape[1] + 1),
    'scratch_ratio': ratio_scratch,
    'sklearn_ratio': pca.explained_variance_ratio_,
    'absolute_difference': np.abs(ratio_scratch - pca.explained_variance_ratio_),
})
PLOT.table(verification, 'Scratch PCA vs scikit-learn PCA', height=360)
print('Maximum explained-variance-ratio difference:', verification['absolute_difference'].max())

**Scratch PCA vs scikit-learn PCA**

Maximum explained-variance-ratio difference: 6.938893903907228e-17


## 16. Scree plot and cumulative explained variance

The scree plot asks how quickly eigenvalues decay. The cumulative curve asks how many components are needed to retain a chosen proportion of total variance.

There is no universally correct threshold. A 90% target is a convention, not a theorem.

In [46]:
components = np.arange(1, X.shape[1] + 1)
ratio = pca.explained_variance_ratio_
cum = np.cumsum(ratio)

p1 = PLOT.line(components, ratio, 'PCA scree plot', 'Principal component', 'Explained variance ratio')
p2 = PLOT.line(components, cum, 'Cumulative explained variance', 'Number of retained components', 'Cumulative explained variance')
p2.add_layout(Span(location=0.90, dimension='width', line_dash='dashed', line_width=2))
show(gridplot([[p1, p2]]))

k90 = int(np.argmax(cum >= 0.90) + 1)
print(f'Components required to reach at least 90% cumulative variance: {k90}')

Components required to reach at least 90% cumulative variance: 8


### Interpretation

A small number of components explaining most variance suggests that the original 13 measurements have substantial redundancy.

But remember:

> **PCA preserves variance, not necessarily predictive information.**

A low-variance direction can still be important for classification or scientific interpretation.

## 17. PCA loadings: what do the components mean?

The eigenvector coefficients are often called **loadings** in practical PCA discussion. Large absolute coefficients indicate variables that strongly contribute to a component direction.

The sign is relative: flipping the sign of every loading in a component represents the same axis.

In [64]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=[f'PC{i}' for i in range(1, X.shape[1] + 1)]
)
show(PLOT.heatmap(loadings.iloc[:, :5], 'PCA loading matrix — first five components', low=-1, high=1))

loading_summary = []
for pc in ['PC1', 'PC2', 'PC3']:
    top = loadings[pc].abs().sort_values(ascending=False).head(5).index
    for feature in top:
        loading_summary.append({'component': pc, 'feature': feature, 'loading': loadings.loc[feature, pc]})
PLOT.table(pd.DataFrame(loading_summary), 'Largest absolute loadings for PC1–PC3', height=330)

**Largest absolute loadings for PC1–PC3**

## 18. PCA score plot

Now project every 13-dimensional observation into the first two principal-component coordinates.

The class labels are used **only for colouring the plot after PCA has been fitted**. PCA itself never saw the labels.

In [65]:
pca_scores = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'PC3': X_pca[:, 2],
    'class_label': df['class_label'].to_numpy(),
    'row': np.arange(len(df)),
})
show(PLOT.grouped_scatter(pca_scores, 'PC1', 'PC2', 'class_label',
                          'PCA score plot — labels shown only for post-hoc interpretation', hover_cols=['row']))

### What should you look for?

- Are classes separated even though PCA ignored labels?
- Which class pairs overlap?
- Are there observations far from the main cloud?
- Does PC1/PC2 separation correspond to the variables with large loadings?

This is an example of **interdependence analysis**: the components were constructed from the structure of $X$ alone.

## 19. A crucial experiment: raw-unit PCA versus standardized PCA

PCA maximises variance. Therefore changing units can change what "large variance" means.

We compare:

1. PCA on the raw measurements;
2. PCA on standardized measurements.

In [48]:
pca_raw = PCA().fit(X_raw)
pca_std = PCA().fit(X_std)

scale_pca = pd.DataFrame({
    'component': np.arange(1, X.shape[1] + 1),
    'raw_unit_ratio': pca_raw.explained_variance_ratio_,
    'standardized_ratio': pca_std.explained_variance_ratio_,
})
PLOT.table(scale_pca, 'Explained variance under two scaling choices', height=360)

# Compare dominant PC1 loadings.
raw_load = pd.Series(pca_raw.components_[0], index=X.columns, name='raw_PC1_loading')
std_load = pd.Series(pca_std.components_[0], index=X.columns, name='std_PC1_loading')
compare_load = pd.concat([raw_load, std_load], axis=1)
compare_load['abs_raw'] = compare_load['raw_PC1_loading'].abs()
compare_load['abs_std'] = compare_load['std_PC1_loading'].abs()
display(compare_load.sort_values('abs_raw', ascending=False).head(8))

**Explained variance under two scaling choices**

,raw_PC1_loading,std_PC1_loading,abs_raw,abs_std
proline,0.9998,0.2868,0.9998,0.2868
magnesium,0.0179,0.1420,0.0179,0.1420
alcalinity_of_ash,-0.0047,-0.2393,0.0047,0.2393
color_intensity,0.0023,-0.0886,0.0023,0.0886
alcohol,0.0017,0.1443,0.0017,0.1443
flavanoids,0.0016,0.4229,0.0016,0.4229
total_phenols,0.0010,0.3947,0.0010,0.3947
od280_od315_of_diluted_wines,0.0007,0.3762,0.0007,0.3762


### Lesson

If one variable has dramatically larger raw variance than the others, raw-unit PCA may mostly track that variable's unit scale.

Standardized PCA instead asks:

> Which combinations explain the most variation **after putting variables on comparable variance scales?**

This experiment is one of the best demonstrations that preprocessing choices are part of the statistical model, not merely implementation details.

# Part V — Classification with Linear Discriminant Analysis

## 20. Classification changes the question

PCA asks:

> Which directions explain variance in $X$?

LDA asks:

> Which directions help distinguish known groups?

Under the classical shared-covariance Gaussian model,

$$
\mathbf X\mid Y=k\sim N_p(\boldsymbol\mu_k,\Sigma).
$$

The discriminant score can be written as

$$
\delta_k(\mathbf x)
=
\mathbf x^T\Sigma^{-1}\boldsymbol\mu_k
-\frac12\boldsymbol\mu_k^T\Sigma^{-1}\boldsymbol\mu_k
+\log\pi_k.
$$

This formula reveals why earlier topics mattered: means, covariance, matrix inversion and Gaussian modelling all return here.

## 21. Train/test discipline

Classification performance must be estimated on data not used to fit the classifier. We therefore use a stratified train/test split.

Scaling is put inside a `Pipeline` so that preprocessing is fitted only from the training data.

In [74]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

lda_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('lda', LinearDiscriminantAnalysis()),
])
lda_pipe.fit(X_train, y_train)
y_pred = lda_pipe.predict(X_test)

metrics = pd.DataFrame([{
    'accuracy': accuracy_score(y_test, y_pred),
    'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
    'macro_f1': f1_score(y_test, y_pred, average='macro'),
}])
PLOT.table(metrics, 'LDA hold-out performance', height=130)
print(classification_report(y_test, y_pred, target_names=[class_map[i] for i in sorted(class_map)]))

**LDA hold-out performance**

              precision    recall  f1-score   support

  cultivar_1       0.95      1.00      0.97        18
  cultivar_2       1.00      0.95      0.98        21
  cultivar_3       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54



## 22. Confusion matrix

A single accuracy number hides which classes are confused. The confusion matrix keeps the class structure visible.

In [75]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=[class_map[i] for i in sorted(class_map)], columns=[class_map[i] for i in sorted(class_map)])
show(PLOT.heatmap(cm_df, 'LDA confusion matrix', low=0, high=max(1, int(cm.max()))))

## 23. LDA projection versus PCA projection

With $K$ classes, classical LDA yields at most $K-1$ discriminant axes. Here $K=3$, so at most two axes are available.

These axes are explicitly chosen using class labels, unlike PCA.

In [76]:
# Fit on all observations only for a descriptive projection plot.
# Performance estimates above remain based on the untouched test set.
scaler_all = StandardScaler()
X_all_std = scaler_all.fit_transform(X)
lda_all = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda_all.fit_transform(X_all_std, y)

lda_scores = pd.DataFrame({
    'LD1': X_lda[:, 0],
    'LD2': X_lda[:, 1],
    'class_label': df['class_label'].to_numpy(),
    'row': np.arange(len(df)),
})

p_pca = PLOT.grouped_scatter(pca_scores, 'PC1', 'PC2', 'class_label', 'PCA: maximize overall variance')
p_lda = PLOT.grouped_scatter(lda_scores, 'LD1', 'LD2', 'class_label', 'LDA: maximize class separation')

show(
    gridplot(
        [[p_pca, p_lda]],
        merge_tools=False,
    )
)

### The conceptual contrast

PCA solves an **unsupervised variance problem**.

LDA solves a **supervised separation problem**.

Therefore a PCA plot that shows weaker class separation does not mean PCA failed. It optimized a different objective.

# Part VI — Clustering

## 24. Classification versus clustering

Classification assumes labels exist during model fitting.

Clustering deliberately ignores labels and asks whether natural groups can be discovered from the feature geometry alone.

For the next experiments, we fit clustering algorithms using only `X_std`. The cultivar labels are used afterward only to evaluate how discovered clusters happen to align with known classes.

## 25. Choosing the number of K-means clusters

For $k=2,\ldots,8$, we compute the silhouette score:

$$
s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}},
$$

where $a(i)$ is average within-cluster dissimilarity and $b(i)$ is the smallest average dissimilarity to another cluster.

Higher silhouette values generally indicate more compact and well-separated clusters.

**Crucially, silhouette does not require true labels.**

In [81]:
rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=30, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_std)
    rows.append({
        'k': k,
        'inertia': km.inertia_,
        'silhouette': silhouette_score(X_std, labels),
        # ARI is post-hoc only; it must NOT be used to choose k in a truly unsupervised setting.
        'posthoc_ARI_vs_known_cultivar': adjusted_rand_score(y, labels),
    })

k_sweep = pd.DataFrame(rows)
PLOT.table(k_sweep, 'K-means sensitivity to number of clusters', height=240)

p_sil = PLOT.line(k_sweep['k'], k_sweep['silhouette'], 'Silhouette score vs k', 'Number of clusters k', 'Silhouette score')
p_inertia = PLOT.line(k_sweep['k'], k_sweep['inertia'], 'K-means inertia vs k', 'Number of clusters k', 'Within-cluster sum of squares')


show(gridplot([[p_sil, p_inertia]]))

**K-means sensitivity to number of clusters**

### Why report ARI but not optimize it?

The Adjusted Rand Index compares discovered clusters with known cultivar labels. That is useful here because the dataset happens to contain ground truth.

But if we select $k$ by maximizing ARI, we have quietly turned an unsupervised problem into a supervised one.

So:

- **silhouette/inertia**: valid unsupervised diagnostics;
- **ARI vs known class**: post-hoc scientific evaluation only.

## 26. K-means with three clusters

We now choose $k=3$ for a controlled comparison with the dataset's three known cultivars. In a genuinely unlabeled application, this choice would need to be justified without using the class labels.

In [58]:
km3 = KMeans(n_clusters=3, n_init=50, random_state=RANDOM_STATE)
cluster_km = km3.fit_predict(X_std)

cluster_plot = pca_scores.copy()
cluster_plot['cluster'] = pd.Series(cluster_km).map(lambda z: f'cluster_{z}')

show(PLOT.grouped_scatter(cluster_plot, 'PC1', 'PC2', 'cluster',
                          'K-means clusters displayed in PCA coordinates'))

cluster_table = pd.crosstab(
    df['class_label'],
    pd.Series(cluster_km, name='cluster'),
    rownames=['known_cultivar'],
    colnames=['discovered_cluster'],
).reset_index()
PLOT.table(cluster_table, 'Post-hoc: discovered clusters vs known cultivars', height=190)
print('Adjusted Rand Index:', adjusted_rand_score(y, cluster_km))

**Post-hoc: discovered clusters vs known cultivars**

Adjusted Rand Index: 0.8974949815093207


### Interpretation caution

Cluster numbers have no inherent meaning. `cluster_0` is not "class 0". Cluster labels can be permuted with no change to the clustering solution.

The crosstab should therefore be interpreted by **pattern**, not by numeric label equality.

## 27. Hierarchical clustering

K-means searches for a fixed number of centroid-based clusters. Hierarchical clustering instead builds a nested sequence of merges.

Ward linkage merges clusters to minimize the increase in within-cluster squared variation.

The dendrogram is a useful multiscale view: cutting at different heights produces different numbers of clusters.

In [82]:
# scipy computes the tree; Bokeh renders the dendrogram segments.
Z = linkage(X_std, method='ward')
ddata = dendrogram(Z, no_plot=True, labels=[str(i) for i in range(len(df))])

p = figure(width=900, height=450, title='Ward hierarchical clustering dendrogram',
           tools='pan,wheel_zoom,box_zoom,reset,save')
for xs, ys in zip(ddata['icoord'], ddata['dcoord']):
    p.line(xs, ys, line_width=1.2)
p.xaxis.axis_label = 'Observations (leaf order; labels hidden for readability)'
p.yaxis.axis_label = 'Ward merge distance'
p.xaxis.major_label_text_font_size = '0pt'
show(p)

agg3 = AgglomerativeClustering(n_clusters=3, linkage='ward')
cluster_h = agg3.fit_predict(X_std)
print('Hierarchical clustering silhouette:', silhouette_score(X_std, cluster_h))
print('Hierarchical clustering post-hoc ARI:', adjusted_rand_score(y, cluster_h))

Hierarchical clustering silhouette: 0.2774439826952265
Hierarchical clustering post-hoc ARI: 0.7899332213582837


## 28. K-means versus hierarchical clustering

Different clustering algorithms encode different assumptions.

| Idea | K-means | Ward hierarchical |
|---|---|---|
| Representation | centroids | merge tree |
| Need $k$ in advance? | yes | tree can be cut later |
| Objective | within-cluster squared distance | greedy minimum variance increase |
| Shape preference | roughly compact/spherical | also variance/distance driven |
| Output | one partition | hierarchy + chosen partition |

Clustering is not simply "run algorithm → obtain truth". The algorithm defines what type of structure it is capable of discovering.

In [54]:
cluster_compare = pd.DataFrame([
    {
        'method': 'KMeans(k=3)',
        'silhouette': silhouette_score(X_std, cluster_km),
        'posthoc_ARI': adjusted_rand_score(y, cluster_km),
    },
    {
        'method': 'Ward hierarchical(k=3)',
        'silhouette': silhouette_score(X_std, cluster_h),
        'posthoc_ARI': adjusted_rand_score(y, cluster_h),
    },
])
PLOT.table(cluster_compare, 'Clustering comparison', height=150)

**Clustering comparison**

# Part VII — Connecting the methods

## 29. One dataset, five different statistical questions

The same $X$ can support very different analyses because each method asks a different question.

| Method | Main question | Uses labels? | Main mathematical object |
|---|---|---:|---|
| covariance analysis | how do variables move together? | no | $S$ / $R$ |
| Hotelling / MANOVA | do population mean vectors differ? | group membership | mean vectors + covariance |
| PCA | which directions explain variance? | no | eigenvectors of covariance/correlation |
| LDA | which directions discriminate known groups? | yes | class means + pooled covariance |
| clustering | what groups emerge from geometry? | no | distances / within-cluster variation |

The major learning objective is therefore not memorising APIs. It is learning to identify **which statistical question you are actually asking**.

## 30. A method-selection decision framework

Use this sequence when faced with a new multivariate problem.

### Step 1 — What is one observation?
Write it explicitly as $\mathbf x_i\in\mathbb R^p$.

### Step 2 — What are the units and scales?
Decide whether standardization is scientifically appropriate.

### Step 3 — What dependence structure exists?
Inspect $S$, $R$, scatterplots and outliers.

### Step 4 — Is your goal inference or structure discovery?

If the goal is comparing populations:

$$
\text{Hotelling / MANOVA-type reasoning}.
$$

If the goal is reducing dimensions:

$$
\text{PCA}.
$$

If labels are known and the goal is prediction:

$$
\text{LDA / classification}.
$$

If labels are unknown and the goal is group discovery:

$$
\text{clustering}.
$$

### Step 5 — Check assumptions
Gaussianity, covariance structure, independence, scaling and sample size all matter.

### Step 6 — Interpret in the original scientific context
A statistically clean component or cluster is useful only if it can be related back to real variables and decisions.

# Part VIII — Additional experiments for deeper understanding

## 31. Experiment: remove correlation structure and see PCA change

We create a synthetic version where each feature is independently shuffled across rows. This approximately preserves each feature's marginal distribution while destroying cross-feature dependence.

Prediction: if correlation is the reason PCA compresses well, then destroying dependence should make explained variance more evenly spread across components.

In [55]:
rng = np.random.default_rng(RANDOM_STATE)
X_shuffled = X_std.copy()
for j in range(X_shuffled.shape[1]):
    rng.shuffle(X_shuffled[:, j])

pca_original = PCA().fit(X_std)
pca_shuffled = PCA().fit(X_shuffled)

compare = pd.DataFrame({
    'component': np.arange(1, X.shape[1] + 1),
    'original': pca_original.explained_variance_ratio_,
    'dependence_destroyed': pca_shuffled.explained_variance_ratio_,
})

p = figure(width=760, height=430, title='PCA depends on cross-feature structure',
           tools='pan,wheel_zoom,reset,save')
p.line(compare['component'], compare['original'], line_width=2, legend_label='original')
p.scatter(compare['component'], compare['original'], size=7)
p.line(compare['component'], compare['dependence_destroyed'], line_width=2, line_dash='dashed', legend_label='independently shuffled')
p.scatter(compare['component'], compare['dependence_destroyed'], size=7, marker='triangle')
p.legend.location = 'top_right'
p.xaxis.axis_label = 'Principal component'
p.yaxis.axis_label = 'Explained variance ratio'
show(p)

### Why this experiment matters

PCA does not magically compress arbitrary data. Compression is possible when variables share structure.

Destroying correlation tends to flatten the eigenvalue spectrum because the variables stop sharing common directions of variation.

This provides an intuitive bridge from **correlation matrix → eigenvalues → dimension reduction**.

## 32. Experiment: what happens to Mahalanobis distance if covariance is ignored?

For standardized data, replacing $S^{-1}$ with the identity matrix reduces Mahalanobis distance to Euclidean distance.

So the difference between the two measures is exactly the correction for covariance geometry.

In [56]:
rank_compare = pd.DataFrame({
    'row': np.arange(len(df)),
    'euclidean_rank': pd.Series(euclid2).rank(ascending=False, method='min').astype(int),
    'mahalanobis_rank': pd.Series(md2).rank(ascending=False, method='min').astype(int),
})
rank_compare['rank_shift'] = (rank_compare['euclidean_rank'] - rank_compare['mahalanobis_rank']).abs()
rank_compare = rank_compare.sort_values('rank_shift', ascending=False)
PLOT.table(rank_compare.head(15), 'Observations whose outlier rank changes most after covariance correction', height=330)

**Observations whose outlier rank changes most after covariance correction**

### Interpretation

Large rank shifts demonstrate why multivariate outlier detection cannot always be replaced by "standardize and compute Euclidean distance".

Two observations can have similar marginal extremeness but very different joint plausibility once feature correlations are considered.

# Part IX — Common mistakes

## 33. Mistakes to avoid in ST4250-style analysis

### Mistake 1 — Interpreting covariance magnitude without considering units
Use correlation when comparing strength across differently scaled measurements.

### Mistake 2 — Running PCA blindly on raw units
Ask whether the scale itself deserves to dominate total variance.

### Mistake 3 — Calling PCA a classifier
PCA has no class-separation objective.

### Mistake 4 — Choosing cluster count using known labels
That leaks supervision into an unsupervised problem.

### Mistake 5 — Treating a small MANOVA p-value as a complete explanation
It detects a joint difference but does not explain which outcomes or directions cause it.

### Mistake 6 — Deleting all high Mahalanobis-distance observations
Outlier scores are diagnostics, not automatic deletion rules.

### Mistake 7 — Forgetting that eigenvector signs are arbitrary
$\mathbf v$ and $-\mathbf v$ define the same PCA axis.

### Mistake 8 — Ignoring covariance assumptions in classical tests/LDA
Shared-covariance methods can be sensitive when population covariance structures differ substantially.

# Part X — Exercises

## 34. Conceptual exercises

1. Why is a covariance matrix always symmetric?
2. Why are all eigenvalues of a valid covariance matrix non-negative?
3. Under what circumstances would covariance-based PCA and correlation-based PCA give similar results?
4. Can two variables be individually normal while the joint vector is not multivariate normal?
5. Why can a point be far in Euclidean distance but modest in Mahalanobis distance?
6. Why is PCA unsupervised even when we colour a PCA plot by known classes afterward?
7. What changes in LDA if class prior probabilities $\pi_k$ are unequal?
8. Why does clustering have an identifiability issue with numeric cluster labels?
9. Why might MANOVA be preferable to four independent ANOVAs on correlated outcomes?
10. Why can dimensionality reduction hurt classification even when it retains 95% of total variance?

## 35. Coding exercises

### Exercise A — Hotelling sensitivity
Repeat the two-sample Hotelling test with different feature subsets. Does the conclusion change?

### Exercise B — PCA threshold
Write a function returning the smallest number of PCs needed for 80%, 90%, 95% and 99% cumulative variance.

### Exercise C — PCA reconstruction
Reconstruct standardized observations using only the first $k$ PCs and measure mean squared reconstruction error as $k$ increases.

### Exercise D — Robust outlier comparison
Compare classical covariance-based Mahalanobis distance with a robust covariance estimator such as `MinCovDet` from scikit-learn.

### Exercise E — LDA assumptions
Compare LDA with Quadratic Discriminant Analysis, which allows class-specific covariance matrices.

### Exercise F — Cluster stability
Run K-means over many random seeds and quantify how stable the partitions are using pairwise Adjusted Rand Index.

### Exercise G — Hierarchical linkage
Compare Ward, complete and average linkage and explain why their dendrograms differ.

### Exercise H — Biplot-style interpretation
Combine PCA scores and scaled loading vectors into one Bokeh figure to see observations and feature directions together.

# Part XI — Final synthesis

## 36. The ST4250 mental model

The course can be remembered through one chain:

$$
\boxed{
\text{random vector}
\rightarrow
(\boldsymbol\mu,\Sigma)
\rightarrow
\text{geometry}
\rightarrow
\text{inference / reduction / grouping}
}
$$

### Mean vector

$$
\boldsymbol\mu
$$

locates the centre.

### Covariance matrix

$$
\Sigma
$$

controls spread, dependence and orientation.

### Eigen-decomposition

$$
\Sigma=V\Lambda V^T
$$

reveals major directions of variation.

### Mahalanobis distance

$$
(\mathbf x-\boldsymbol\mu)^T\Sigma^{-1}(\mathbf x-\boldsymbol\mu)
$$

measures distance relative to covariance geometry.

### Hotelling / MANOVA

compare mean vectors jointly rather than one outcome at a time.

### PCA

uses covariance geometry to reduce dimension without labels.

### LDA

uses class means and covariance to discriminate known populations.

### Clustering

uses multivariate geometry to discover groups without known labels.

If you understand why the **same mean/covariance/eigenvalue ideas reappear across all these methods**, you understand the unifying logic of multivariate statistical analysis rather than a list of disconnected algorithms.

## 37. Suggested second notebook / extensions

The following are natural extensions after the classical ST4250 core covered here:

- robust covariance estimation;
- Quadratic Discriminant Analysis;
- regularized discriminant analysis;
- factor analysis;
- canonical correlation analysis;
- multidimensional scaling;
- Gaussian mixture clustering;
- model-based clustering;
- sparse PCA;
- high-dimensional covariance estimation;
- permutation MANOVA / non-parametric multivariate testing.

These are best treated as **extensions** rather than assuming every one is guaranteed to be part of a particular ST4250 offering.